# 배치 정규화

**역할**

레이어별 활성값을 배치 차원에서 정규화 → 학습을 안정화, 더 큰 학습률 허용, 수렴 가속.

약한 정규화 효과로 과적합도 조금 완화.

In [ ]:
# 사용법

nn.Sequential(
  nn.Conv2d(Cin, Cout, 3, padding=1),
  nn.BatchNorm2d(Cout),
  nn.ReLU(inplace=True)
)
# 학습/평가 모드 전환
model.train();  # BN이 배치 통계 사용
model.eval();   # BN이 이동평균(러닝 스탯) 사용

# 학습률 스케줄링

**역할**

학습 초반엔 크게, 후반엔 작게 → 빠른 수렴 + 더 나은 최종 성능.

폭주/진동 방지, 지역 최적해 탈출 도움.


**사용 타이밍**

Step/Multistep: 특정 에폭마다 LR×γ. 간단, ResNet류 전형.

Cosine Annealing(+ Warmup): 최근 디폴트급. 큰 배치/AdamW와 궁합 좋음.

One-Cycle: 짧은 학습에서 강력.

ReduceLROnPlateau: 검증 성능 정체 시 자동 감쇠(전이학습에 편함).

In [ ]:
# 사용법

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
# warmup + cosine(예: timm/scheduler 또는 transformers의 get_cosine_schedule_with_warmup)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

for epoch in range(epochs):
    train_one_epoch(...)
    scheduler.step()

# 데이터 증강

**역할**

입력 다양화로 일반화 성능↑(오버피팅 ↓).

입력 불변성(회전, 밝기, 시간 이동 등)을 모델이 학습하도록 유도.

**대표 기법**

비전: RandomCrop/Resize, Flip, ColorJitter, RandomErasing(Cutout), Mixup/CutMix, AutoAugment/TrivialAugment.

오디오: Time shift, Noise, SpecAugment(time/freq masking).

텍스트: 백-번역, 동의어 치환, 랜덤 마스킹(프리트레인).

In [ ]:
# 사용법

train_tf = torchvision.transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
])
val_tf = torchvision.transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# 가중치 초기화

**역할**

신호/그라디언트가 레이어를 지나며 폭주·소실되지 않도록 분산을 맞춤.

초기 학습 안정성/속도에 큰 영향.

In [ ]:
# 사용법

def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)
    if isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)   # gamma
        nn.init.zeros_(m.bias)    # beta

model.apply(init_weights)